In [1]:
import json
import numpy as np
import pandas as pd

def mark_to_numeric(mark):
    if pd.isna(mark):
        return np.nan

    mark = str(mark).strip()

    if ":" in mark:
        try:
            parts = [float(part) for part in mark.split(":")]
        except ValueError:
            return np.nan

        if len(parts) == 2:
            minutes, seconds = parts
            return minutes * 60 + seconds

        if len(parts) == 3:
            hours, minutes, seconds = parts
            return hours * 3600 + minutes * 60 + seconds

        return np.nan

    try:
        return float(mark)
    except ValueError:
        return np.nan

In [2]:
df = pd.read_csv('data.csv', sep=";")
df

,Rank,Mark,Competitor,DOB,Nat,Pos,Venue,Date,Results Score,Mark [meters or seconds],Event,Wind,Sex
0,1,3:43.13,Hicham EL GUERROUJ,1974-09-14,MAR,1,"Stadio Olimpico, Roma (ITA)",1999-07-07,1292,223.1,One Mile,NaN,male
1,2,3:43.40,Noah NGENY,1978-11-02,KEN,2,"Stadio Olimpico, Roma (ITA)",1999-07-07,1288,223.4,One Mile,NaN,male
2,3,3:44.39,Noureddine MORCELI,1970-02-28,ALG,1,Rieti (ITA),1993-09-05,1275,224.3,One Mile,NaN,male
3,4,3:44.60,Hicham EL GUERROUJ,1974-09-14,MAR,1,Nice (FRA),1998-07-16,1272,224.6,One Mile,NaN,male
4,5,3:44.90,Hicham EL GUERROUJ,1974-09-14,MAR,1,Oslo (NOR),1997-07-04,1268,224.9,One Mile,NaN,male
...,...,...,...,...,...,...,...,...,...,...,...,...,...
461517,2206,6.75,Claire BRYANT,2001-08-25,USA,3,"Percy Beard Track, Gainesville, FL (USA)",2023-04-14,1163,6.7,Long Jump,1.8,female
461518,2206,6.75,Sumire HATA,1996-05-04,JPN,1,"Prefectural Shizuoka Stadium, Fukuroi (JPN)",2023-05-03,1163,6.7,Long Jump,2.0,female
461519,2206,6.75,Fátima DIAME,1996-09-22,ESP,1,"Helmántico, Salamanca (ESP)",2023-06-03,1179,6.7,Long Jump,-2.7,female
461520,2206,6.75,Diana LESTI,1998-03-30,HUN,1,"Lantos Mihály Sportközpont, Budapest (HUN)",2023-06-08,1164,6.7,Long Jump,-0.3,female


In [3]:
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

In [4]:
df["DOB"] = pd.to_datetime(df["DOB"], errors="coerce")

Différence performance moyenne vs record

In [5]:
import os
import json
import numpy as np
import pandas as pd

df = pd.read_csv("data.csv", sep=";")

df["Performance"] = df["Mark [meters or seconds]"]

field_events = {
    "Long Jump", "Triple Jump", "High Jump", "Pole Vault",
    "Shot Put", "Discus Throw", "Hammer Throw", "Javelin Throw"
}

df["Event_Type"] = np.where(df["Event"].isin(field_events), "field", "track")
df["Unit"] = np.where(df["Event_Type"].eq("field"), "m", "s")

normal_perf = (
    df.groupby(["Event", "Sex", "Event_Type", "Unit"])["Performance"]
    .median()
    .reset_index(name="normal_perf")
)

records = []
for (event, sex), group in df.groupby(["Event", "Sex"]):
    event_type = group["Event_Type"].iloc[0]
    unit = group["Unit"].iloc[0]

    median_perf = group["Performance"].median()

    if event_type == "track":
        record = group["Performance"].min()
        gap = median_perf - record
    else:
        record = group["Performance"].max()
        gap = record - median_perf

    records.append({
        "Event": event,
        "Sex": sex,
        "record_perf": record,
        "gap": gap,
        "Unit": unit
    })

record_perf = pd.DataFrame(records)

df_diff_perf_record = normal_perf.merge(
    record_perf,
    on=["Event", "Sex", "Unit"]
)

plot_data = df_diff_perf_record.sort_values("gap", ascending=False).head(15)

data_dict = {
    "labels": (plot_data["Event"] + " " + plot_data["Sex"]).tolist(),
    "gap": plot_data["gap"].round(2).tolist(),
    "normal_perf": plot_data["normal_perf"].round(2).tolist(),
    "record_perf": plot_data["record_perf"].round(2).tolist(),
    "unit": plot_data["Unit"].tolist()
}

os.makedirs("visualizations_data/gap_record", exist_ok=True)

with open("visualizations_data/gap_record/data.json", "w") as f:
    json.dump(data_dict, f, indent=2)

Évolution des records

In [6]:
df["Performance"] = df["Mark"].apply(mark_to_numeric)

duels entre ⅔ pers de ≠ époques

In [7]:
import os
import json
import pandas as pd

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Year"] = df["Date"].dt.year
df["Performance"] = df["Mark"].apply(mark_to_numeric)

df_duel = df.dropna(subset=["Event", "Sex", "Competitor", "Performance", "Year"]).copy()
year_stats = (
    df_duel
    .groupby(["Event","Sex","Year","Event_Type","Unit"], as_index=False)
    .agg(
        mean=("Performance","mean"),
        record=("Performance","min")
    )
)
all_data = {}

for (event, sex), group in year_stats.groupby(["Event","Sex"]):

    yearly_data = {}

    for _, row in group.iterrows():
        year = int(row["Year"])

        yearly_data[str(year)] = {
            "record": round(row["record"],2),
            "mean": round(row["mean"],2)
        }

    all_data[f"{event}___{sex}"] = {
        "event": event,
        "sex": sex,
        "unit": group["Unit"].iloc[0],
        "event_type": group["Event_Type"].iloc[0],
        "years": sorted(group["Year"].tolist()),
        "yearly_data": yearly_data
    }

os.makedirs("visualizations_data/duel_athletes", exist_ok=True)

with open("visualizations_data/duel_athletes/data.json", "w") as f:
    json.dump(all_data, f, indent=2)

print("done")

done
